# Phase 2 — Feature Engineering

Computes weekly features (sections 1-6) over the 4-week observation window and saves
`weekly_features.parquet` for downstream validation and modeling.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import datetime

spark = SparkSession.builder \
    .appName("gfn-feature-engineering") \
    .master("local[*]") \
    .getOrCreate()

users = spark.read.parquet("/home/spark/work/data/raw/users.parquet")
sessions = spark.read.parquet("/home/spark/work/data/raw/session_logs.parquet")
games = spark.read.parquet("/home/spark/work/data/raw/game_catalog.parquet")
sub_events = spark.read.parquet("/home/spark/work/data/raw/subscription_events.parquet")
payments = spark.read.parquet("/home/spark/work/data/raw/payments.parquet")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/27 03:55:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


##### Temporal constants

In [2]:
# Temporal windows
DATA_START    = datetime.date(2024, 1, 1)    # Week 1 start
BASELINE_END  = datetime.date(2024, 1, 29)   # Week 5 start = baseline end
OBS_START     = datetime.date(2024, 1, 29)   # Observation window start (week 5)
OBS_END       = datetime.date(2024, 2, 26)   # Observation window end (week 9 start, exclusive)
BASELINE_WEEKS = 4

In [3]:
# ── Shared base: obs_sessions with week_num and duration ──
obs_sessions = sessions.filter(
    (F.col("start_time").cast("date") >= F.lit(OBS_START)) &
    (F.col("start_time").cast("date") < F.lit(OBS_END))
).withColumn(
    "week_num",
    (F.datediff(F.col("start_time").cast("date"), F.lit(OBS_START)) / 7).cast("int") + 1
).withColumn(
    "duration_min",
    (F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60.0
)

##### Session Patterns

In [4]:
# ══════════════════════════════════════════════
# Section 1: Session Patterns
# Output: session_features (user_id, week_num, 6 features)
# ══════════════════════════════════════════════

# Core aggregations
session_patterns = obs_sessions.groupBy("user_id", "week_num").agg(
    F.count("session_id").alias("weekly_session_count"),
    F.avg("duration_min").alias("avg_session_duration_min"),
    F.sum("duration_min").alias("total_playtime_min"),
    F.avg(
        F.when(F.hour("start_time").between(19, 23), 1).otherwise(0)
    ).alias("peak_hour_ratio"),
    F.avg(
        F.when(F.dayofweek("start_time").isin(1, 7), 1).otherwise(0)
    ).alias("weekend_ratio"),
)

# Session regularity: std of inter-session gaps (computed separately due to lag)
w_session_order = Window.partitionBy("user_id", "week_num").orderBy("start_time")

session_regularity = obs_sessions.withColumn(
    "prev_end", F.lag("end_time").over(w_session_order)
).withColumn(
    "inter_session_gap_min",
    (F.unix_timestamp("start_time") - F.unix_timestamp("prev_end")) / 60.0
).filter(
    F.col("inter_session_gap_min").isNotNull()
).groupBy("user_id", "week_num").agg(
    F.stddev("inter_session_gap_min").alias("session_regularity")
)

# Combine into section output
session_features = session_patterns.join(
    session_regularity, on=["user_id", "week_num"], how="left"
).fillna(0, subset=["session_regularity"])

##### Population Normalization

Hot event weeks (1.3-1.8x activity boost on all users) distort week-over-week trends.
Normalize `weekly_session_count` and `total_playtime_min` by their population medians
per week so that trend features (wow_change, vs_baseline) reflect individual behavior,
not global events.

In [5]:
# ══════════════════════════════════════════════
# Section 1.5: Population-Level Normalization
# Compute medians from active users only (before zero-fill scaffold)
# ══════════════════════════════════════════════

pop_medians = session_features.groupBy("week_num").agg(
    F.expr("percentile_approx(weekly_session_count, 0.5)").alias("pop_median_session_count"),
    F.expr("percentile_approx(total_playtime_min, 0.5)").alias("pop_median_playtime_min"),
)

pop_medians.orderBy("week_num").show()

session_features = session_features.join(
    pop_medians, on="week_num", how="left"
).withColumn(
    "weekly_session_count_norm",
    F.col("weekly_session_count") / F.col("pop_median_session_count")
).withColumn(
    "total_playtime_min_norm",
    F.col("total_playtime_min") / F.col("pop_median_playtime_min")
).drop("pop_median_session_count", "pop_median_playtime_min")

+--------+------------------------+-----------------------+
|week_num|pop_median_session_count|pop_median_playtime_min|
+--------+------------------------+-----------------------+
|       1|                       5|      362.7166666666667|
|       2|                       5|                  362.0|
|       3|                       4|     309.29999999999995|
|       4|                       7|                  518.3|
+--------+------------------------+-----------------------+



##### Engagement Decay

In [6]:
# ══════════════════════════════════════════════
# Section 2: Engagement Decay
# Input dependency: session_features (for wow_change and baseline ratios)
# Output: engagement_features (user_id, week_num, 6 features)
# ══════════════════════════════════════════════

# --- Scaffold: ensure every user has all 4 weeks ---
all_user_weeks = obs_sessions.select("user_id").distinct().crossJoin(
    spark.createDataFrame([(i,) for i in range(1, 5)], ["week_num"])
)

session_features_filled = all_user_weeks.join(
    session_features, on=["user_id", "week_num"], how="left"
).fillna(0, subset=[
    "weekly_session_count", "total_playtime_min",
    "avg_session_duration_min", "peak_hour_ratio",
    "weekend_ratio", "session_regularity",
    "weekly_session_count_norm", "total_playtime_min_norm",
])

# --- 2a. Week-over-week change (using normalized values) ---
# Normalization removes hot-week effects between consecutive weeks
wow_window = Window.partitionBy("user_id").orderBy("week_num")

wow_base = session_features_filled.select(
    "user_id", "week_num",
    "weekly_session_count_norm", "total_playtime_min_norm"
).withColumn(
    "prev_session_count", F.lag("weekly_session_count_norm").over(wow_window)
).withColumn(
    "prev_playtime", F.lag("total_playtime_min_norm").over(wow_window)
)

wow_features = wow_base.select(
    "user_id", "week_num",
    F.when(
        F.col("prev_session_count").isNull() | (F.col("prev_session_count") == 0),
        F.lit(None)
    ).otherwise(
        (F.col("weekly_session_count_norm") - F.col("prev_session_count"))
        / F.col("prev_session_count")
    ).alias("session_count_wow_change"),

    F.when(
        F.col("prev_playtime").isNull() | (F.col("prev_playtime") == 0),
        F.lit(None)
    ).otherwise(
        (F.col("total_playtime_min_norm") - F.col("prev_playtime"))
        / F.col("prev_playtime")
    ).alias("playtime_wow_change"),
)

# --- 2b. Baseline comparison (using raw values) ---
# vs_baseline is already a per-user relative measure (my activity vs my own history),
# so raw counts are appropriate here. Hot weeks affect only the obs side,
# but the ratio still reflects meaningful individual-level change.
baseline_sessions = sessions.filter(
    (F.col("start_time").cast("date") >= F.lit(DATA_START)) &
    (F.col("start_time").cast("date") < F.lit(BASELINE_END))
).withColumn(
    "duration_min",
    (F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60.0
)

user_baseline = baseline_sessions.groupBy("user_id").agg(
    (F.count("session_id") / F.lit(BASELINE_WEEKS)).alias("baseline_avg_session_count"),
    (F.sum("duration_min") / F.lit(BASELINE_WEEKS)).alias("baseline_avg_playtime"),
)

baseline_ratios = session_features_filled.select(
    "user_id", "week_num",
    "weekly_session_count", "total_playtime_min"
).join(user_baseline, on="user_id", how="left")

baseline_features = baseline_ratios.select(
    "user_id", "week_num",
    F.when(
        F.col("baseline_avg_session_count").isNull() | (F.col("baseline_avg_session_count") == 0),
        F.lit(None)
    ).otherwise(
        F.col("weekly_session_count") / F.col("baseline_avg_session_count")
    ).alias("session_count_vs_baseline"),

    F.when(
        F.col("baseline_avg_playtime").isNull() | (F.col("baseline_avg_playtime") == 0),
        F.lit(None)
    ).otherwise(
        F.col("total_playtime_min") / F.col("baseline_avg_playtime")
    ).alias("playtime_vs_baseline"),
)

# --- 2c. Longest inactive days (unchanged) ---
obs_dates = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{OBS_START}'),
        date_sub(to_date('{OBS_END}'), 1),
        interval 1 day
    )) AS date
""").withColumn(
    "week_num",
    (F.datediff(F.col("date"), F.lit(OBS_START)) / 7).cast("int") + 1
)

obs_user_ids = obs_sessions.select("user_id").distinct()
user_date_scaffold = obs_user_ids.crossJoin(obs_dates)

session_dates = obs_sessions.select(
    "user_id",
    F.col("start_time").cast("date").alias("date")
).distinct().withColumn("had_session", F.lit(1))

daily_activity = user_date_scaffold.join(
    session_dates, on=["user_id", "date"], how="left"
).fillna(0, subset=["had_session"])

streak_window = Window.partitionBy("user_id").orderBy("date")
daily_activity = daily_activity.withColumn(
    "session_cumsum", F.sum("had_session").over(streak_window)
)

inactive_streaks = daily_activity.filter(F.col("had_session") == 0)
streak_group_window = Window.partitionBy("user_id", "session_cumsum").orderBy("date")
inactive_streaks = inactive_streaks.withColumn(
    "streak_len", F.row_number().over(streak_group_window)
)

longest_inactive = inactive_streaks.groupBy("user_id", "week_num").agg(
    F.max("streak_len").alias("longest_inactive_days")
)

# --- 2d. Combine all Section 2 features ---
engagement_features = wow_features \
    .join(baseline_features, on=["user_id", "week_num"], how="outer") \
    .join(longest_inactive,  on=["user_id", "week_num"], how="outer") \
    .fillna(0, subset=["longest_inactive_days"])

##### Streaming Quality

In [7]:
# ══════════════════════════════════════════════
# Section 3: Streaming Quality
# Output: streaming_features (user_id, week_num, 8 features)
# ══════════════════════════════════════════════

streaming_features = obs_sessions.groupBy("user_id", "week_num").agg(
    F.avg("avg_latency_ms").alias("avg_latency"),
    F.avg("avg_fps").alias("avg_fps"),
    F.avg("total_frame_drops").alias("frame_drop_rate"),
    F.avg("disconnect_count").alias("disconnect_rate"),
    F.avg("avg_bitrate_mbps").alias("avg_bitrate"),
    F.avg("avg_jitter_ms").alias("avg_jitter"),
    F.avg("packet_loss_rate").alias("packet_loss_avg"),
    F.avg(
        F.when(F.col("exit_type").isin("crash", "disconnect", "timeout"), 1).otherwise(0)
    ).alias("crash_exit_ratio"),
)

##### Game Diversity

In [8]:
# ══════════════════════════════════════════════
# Section 4: Game Diversity
# Output: game_features (user_id, week_num, 4 features)
# ══════════════════════════════════════════════

# --- 4a. unique_games_played & top_game_concentration ---
game_playtime = obs_sessions.groupBy("user_id", "week_num", "game_id").agg(
    F.sum("duration_min").alias("game_playtime"),
    F.count("session_id").alias("game_sessions"),
)

# Top game concentration: % of playtime on most-played game
w_game = Window.partitionBy("user_id", "week_num")

game_concentration = game_playtime.withColumn(
    "total_playtime", F.sum("game_playtime").over(w_game)
).withColumn(
    "playtime_share", F.col("game_playtime") / F.col("total_playtime")
)

game_basic = game_concentration.groupBy("user_id", "week_num").agg(
    F.countDistinct("game_id").alias("unique_games_played"),
    F.max("playtime_share").alias("top_game_concentration"),
)

In [9]:
# --- 4b. genre_entropy ---
# Shannon entropy of genre distribution per user per week
# H = -Σ p(g) * log2(p(g))

game_with_genre = obs_sessions.join(
    games.select("game_id", "genre"), on="game_id", how="left"
)

genre_counts = game_with_genre.groupBy("user_id", "week_num", "genre").agg(
    F.count("session_id").alias("genre_sessions")
)

w_genre = Window.partitionBy("user_id", "week_num")

genre_probs = genre_counts.withColumn(
    "total_sessions", F.sum("genre_sessions").over(w_genre)
).withColumn(
    "p", F.col("genre_sessions") / F.col("total_sessions")
).withColumn(
    "neg_p_log2_p", -F.col("p") * F.log2(F.col("p"))
)

genre_entropy = genre_probs.groupBy("user_id", "week_num").agg(
    F.sum("neg_p_log2_p").alias("genre_entropy")
)

In [10]:
# --- 4c. new_game_trial_rate ---
# % of sessions on games not played in any prior week within obs window
# week 1 = null (no prior history in obs window to compare)

# All games each user has played up to (but not including) each week
w_cumulative = Window.partitionBy("user_id").orderBy("week_num") \
    .rowsBetween(Window.unboundedPreceding, -1)

# Step 1: Collect distinct games per user per week
user_week_games = obs_sessions.groupBy("user_id", "week_num").agg(
    F.collect_set("game_id").alias("current_week_games")
)

# Step 2: Collect all games played in prior weeks
# Use a self-join approach: for each (user, week), find all games from earlier weeks
prior_games = obs_sessions.select("user_id", "week_num", "game_id").distinct()

prior_games_agg = prior_games.alias("a").join(
    prior_games.alias("b"),
    (F.col("a.user_id") == F.col("b.user_id")) &
    (F.col("a.week_num") > F.col("b.week_num"))
).groupBy(
    F.col("a.user_id").alias("user_id"),
    F.col("a.week_num").alias("week_num")
).agg(
    F.collect_set(F.col("b.game_id")).alias("prior_games")
)

# Step 3: Compute new game trial rate
new_game_rate = user_week_games.join(
    prior_games_agg, on=["user_id", "week_num"], how="left"
).withColumn(
    "new_games",
    F.when(
        F.col("prior_games").isNull(),
        F.lit(None)  # week 1: no prior data → null
    ).otherwise(
        F.size(F.array_except(F.col("current_week_games"), F.col("prior_games")))
    )
).withColumn(
    "new_game_trial_rate",
    F.when(
        F.col("new_games").isNull(),
        F.lit(None)
    ).otherwise(
        F.col("new_games") / F.size(F.col("current_week_games"))
    )
).select("user_id", "week_num", "new_game_trial_rate")

In [11]:
# --- 4d. Combine all Section 4 features ---
game_features = game_basic \
    .join(genre_entropy,  on=["user_id", "week_num"], how="outer") \
    .join(new_game_rate,  on=["user_id", "week_num"], how="outer")

##### Playtime Volatility

In [12]:
# ══════════════════════════════════════════════
# Section 5: Playtime Volatility
# Output: volatility_features (user_id, week_num, 3 features)
# Dependencies: obs_sessions, obs_dates (from Section 2)
# ══════════════════════════════════════════════

# --- 5a. Daily playtime (including 0-session days) ---
# Sum playtime per user per day from obs_sessions
daily_playtime = obs_sessions.groupBy(
    "user_id",
    F.col("start_time").cast("date").alias("date")
).agg(
    F.sum("duration_min").alias("daily_playtime_min")
)

# Reuse the user × date scaffold from Section 2
# Fill missing days with 0 playtime — critical for churn signal
daily_playtime_filled = user_date_scaffold.join(
    daily_playtime, on=["user_id", "date"], how="left"
).fillna(0, subset=["daily_playtime_min"])

# --- 5b. daily_playtime_std & daily_playtime_cv (per user per week) ---
daily_volatility = daily_playtime_filled.groupBy("user_id", "week_num").agg(
    F.stddev("daily_playtime_min").alias("daily_playtime_std"),
    F.avg("daily_playtime_min").alias("daily_playtime_mean"),
)

# CV = std / mean (null when mean = 0 to avoid division by zero)
daily_volatility = daily_volatility.withColumn(
    "daily_playtime_cv",
    F.when(
        F.col("daily_playtime_mean") == 0, F.lit(None)
    ).otherwise(
        F.col("daily_playtime_std") / F.col("daily_playtime_mean")
    )
).drop("daily_playtime_mean")

# --- 5c. session_duration_std (per user per week, sessions only) ---
# This one does NOT include 0-session days — it's about variance
# across actual sessions within that week
session_dur_std = obs_sessions.groupBy("user_id", "week_num").agg(
    F.stddev("duration_min").alias("session_duration_std")
)

# --- 5d. Combine all Section 5 features ---
volatility_features = daily_volatility \
    .join(session_dur_std, on=["user_id", "week_num"], how="outer")

##### Subscription & Payment

In [13]:
# ══════════════════════════════════════════════
# Section 6: Subscription & Payment
# Output: sub_pay_features (user_id, 10 features) — NOT per-week
# These are aggregated over the obs window or are static user attributes
# ══════════════════════════════════════════════

# --- 6a. User-level static features ---
user_static = users.select(
    "user_id",
    "subscription_tier",
    "signup_date",
).withColumn(
    "current_tier_numeric",
    F.when(F.col("subscription_tier") == "free", 0)
     .when(F.col("subscription_tier") == "priority", 1)
     .when(F.col("subscription_tier") == "ultimate", 2)
).withColumn(
    # Days from signup to obs window start
    "days_since_signup",
    F.datediff(F.lit(OBS_START), F.col("signup_date"))
).select("user_id", "current_tier_numeric", "days_since_signup")

In [14]:
# --- 6b. Subscription event features (obs window only) ---
obs_sub_events = sub_events.filter(
    (F.col("event_date") >= F.lit(OBS_START)) &
    (F.col("event_date") < F.lit(OBS_END))
)

sub_features = obs_sub_events.groupBy("user_id").agg(
    F.count("event_id").alias("tier_changes_count"),
    F.max(
        F.when(F.col("event_type") == "downgrade", 1).otherwise(0)
    ).alias("has_downgraded"),
)

In [15]:
# --- 6c. Payment features (obs window only) ---
obs_payments = payments.filter(
    (F.col("payment_date") >= F.lit(OBS_START)) &
    (F.col("payment_date") < F.lit(OBS_END))
)

payment_agg = obs_payments.groupBy("user_id").agg(
    # Total spend (successful payments only)
    F.sum(
        F.when(F.col("status") == "success", F.col("amount_usd")).otherwise(0)
    ).alias("total_spend_last_4w"),

    # Payment count (all statuses — activity indicator)
    F.count("payment_id").alias("payment_count_last_4w"),

    # Failed & refund counts (churn signals)
    F.sum(
        F.when(F.col("status") == "failed", 1).otherwise(0)
    ).alias("failed_payment_count"),
    F.sum(
        F.when(F.col("status") == "refunded", 1).otherwise(0)
    ).alias("refund_count"),

    # Days since last successful payment (recency)
    F.datediff(
        F.lit(OBS_END),
        F.max(F.when(F.col("status") == "success", F.col("payment_date")))
    ).alias("days_since_last_payment"),
)

In [16]:
# --- 6d. Payment frequency change vs baseline ---
baseline_payment_count = payments.filter(
    (F.col("payment_date") >= F.lit(DATA_START)) &
    (F.col("payment_date") < F.lit(BASELINE_END))
).groupBy("user_id").agg(
    F.count("payment_id").alias("baseline_payment_count")
)

obs_payment_count = obs_payments.groupBy("user_id").agg(
    F.count("payment_id").alias("obs_payment_count")
)

payment_freq_change = obs_payment_count.join(
    baseline_payment_count, on="user_id", how="outer"
).fillna(0, subset=["obs_payment_count", "baseline_payment_count"]) \
.withColumn(
    "payment_frequency_change",
    F.when(
        F.col("baseline_payment_count") == 0, F.lit(None)
    ).otherwise(
        (F.col("obs_payment_count") - F.col("baseline_payment_count"))
        / F.col("baseline_payment_count")
    )
).select("user_id", "payment_frequency_change")

In [17]:
# --- 6e. Combine all Section 6 features ---
sub_pay_features = user_static \
    .join(sub_features,       on="user_id", how="left") \
    .join(payment_agg,        on="user_id", how="left") \
    .join(payment_freq_change, on="user_id", how="left") \
    .fillna(0, subset=[
        "tier_changes_count", "has_downgraded",
        "total_spend_last_4w", "payment_count_last_4w",
        "failed_payment_count", "refund_count",
    ])
# Note: days_since_last_payment and payment_frequency_change
# stay null for users with zero payments — meaningful signal

# Broadcast to all 4 weeks (these are user-level, not weekly)
week_scaffold = spark.createDataFrame([(i,) for i in range(1, 5)], ["week_num"])
payment_features = sub_pay_features.crossJoin(week_scaffold)

##### Final Join

In [18]:
# ══════════════════════════════════════════════
# Final: Merge all feature sections
# ══════════════════════════════════════════════

weekly_features = session_features_filled \
    .join(engagement_features, on=["user_id", "week_num"], how="outer") \
    .join(streaming_features,  on=["user_id", "week_num"], how="outer") \
    .join(game_features,       on=["user_id", "week_num"], how="outer") \
    .join(volatility_features, on=["user_id", "week_num"], how="outer") \
    .join(payment_features,    on=["user_id", "week_num"], how="outer")  


zero_fill_cols = [
    "weekly_session_count", "avg_session_duration_min", "total_playtime_min",
    "weekly_session_count_norm", "total_playtime_min_norm",
    "peak_hour_ratio", "weekend_ratio", "session_regularity",
    "longest_inactive_days", "daily_playtime_std",
]
weekly_features = weekly_features.fillna(0, subset=zero_fill_cols)

# Materialize to avoid recomputing the full DAG during parquet write
weekly_features.cache()

print(f"weekly_features: {weekly_features.count():,} rows, {len(weekly_features.columns)} cols")
print(f"Columns: {weekly_features.columns}")

26/03/27 03:56:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/27 03:56:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/27 03:56:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/27 03:56:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/27 03:56:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/27 03:56:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/27 03:56:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/27 03:56:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but re

weekly_features: 200,000 rows, 40 cols
Columns: ['user_id', 'week_num', 'weekly_session_count', 'avg_session_duration_min', 'total_playtime_min', 'peak_hour_ratio', 'weekend_ratio', 'session_regularity', 'weekly_session_count_norm', 'total_playtime_min_norm', 'session_count_wow_change', 'playtime_wow_change', 'session_count_vs_baseline', 'playtime_vs_baseline', 'longest_inactive_days', 'avg_latency', 'avg_fps', 'frame_drop_rate', 'disconnect_rate', 'avg_bitrate', 'avg_jitter', 'packet_loss_avg', 'crash_exit_ratio', 'unique_games_played', 'top_game_concentration', 'genre_entropy', 'new_game_trial_rate', 'daily_playtime_std', 'daily_playtime_cv', 'session_duration_std', 'current_tier_numeric', 'days_since_signup', 'tier_changes_count', 'has_downgraded', 'total_spend_last_4w', 'payment_count_last_4w', 'failed_payment_count', 'refund_count', 'days_since_last_payment', 'payment_frequency_change']


In [19]:
# Save weekly features to parquet for downstream use (validation, modeling)
OUTPUT_PATH = "/home/spark/work/data/processed/weekly_features.parquet"
weekly_features.write.mode("overwrite").parquet(OUTPUT_PATH)
print(f"Saved weekly_features to {OUTPUT_PATH}")

26/03/27 03:56:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/27 03:56:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/27 03:56:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/27 03:56:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/27 03:56:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/27 03:56:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/27 03:56:24 WARN MemoryManager: Total allocation exceeds 95.00%

Saved weekly_features to /home/spark/work/data/processed/weekly_features.parquet
